# পাঠ ১১ - এজেন্ট-টু-এজেন্ট (A2A) প্রটোকল


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A প্রটোকল কী?

এই **এজেন্ট-টু-এজেন্ট (A2A) প্রটোকল** একটি খোলা স্ট্যান্ডার্ড যা এআই এজেন্টদের যোগাযোগ,
একে অপরকে আবিষ্কার করতে, এবং সহযোগিতা করতে — এমনকি যখন তারা ভিন্ন ফ্রেমওয়ার্কে নির্মিত বা
বিভিন্ন সার্ভিসে হোস্ট করা থাকে।

মূল ধারণা:

- **আবিষ্কার** – এজেন্টরা তাদের সক্ষমতাগুলি বর্ণনা করে এমন একটি *এজেন্ট কার্ড* প্রকাশ করে, যা
  যাতে অন্যান্য এজেন্টরা (বা অর্কেস্ট্রেটররা) একটি কাজের জন্য সঠিক বিশেষজ্ঞ খুঁজে পেতে সহজ হয়।
- **বার্তা আদানপ্রদান** – এজেন্টরা একটি সাধারণ প্রটোকলের মাধ্যমে কাঠামোবদ্ধ বার্তা বিনিময় করে, তাই একটি
  অন্য এজেন্টের কাছ থেকে আসা অনুরোধকে বোঝা এবং পূরণ করা যেতে পারে, তার অভ্যন্তরীণ
  বাস্তবায়ন নির্বিশেষে।
- **কাজের লাইফসাইকেল** – A2A নির্ধারণ করে অবস্থাগুলি যেমন *জমা করা হয়েছে*, *চলমান*, *সম্পন্ন*, এবং
  *ব্যর্থ*, অর্কেস্ট্রেটরকে নিযুক্ত কাজ কীভাবে অগ্রসর হচ্ছে তার সম্পূর্ণ দৃশ্যমানতা প্রদান করে।

এই পাঠে আমরা A2A-ধাঁচের সহযোগিতা অনুকরণ করি তিনটি বিশেষায়িত ভ্রমণ এজেন্টকে
একটি ওয়ার্কফ্লোতে সংযুক্ত করে যেখানে প্রতিটি এজেন্ট তার দক্ষতা প্রদান করে এবং ফলাফল পরবর্তী এজেন্টকে পাঠায়।


## বিশেষায়িত ভ্রমণ এজেন্ট তৈরি


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ওয়ার্কফ্লোর মাধ্যমে মাল্টি-এজেন্ট সহযোগিতা

আমরা তিনটি এজেন্টকে একটি ক্রমানুসারী ওয়ার্কফ্লোতে সংযুক্ত করি যা A2A বার্তা-প্রেরণের প্রতিফলন করে:

1. **CurrencyExchangeAgent** ব্যবহারকারীর অনুরোধ গ্রহণ করে এবং মুদ্রা নির্দেশনা প্রদান করে।
2. **ActivityPlannerAgent** সমৃদ্ধ প্রসঙ্গ গ্রহণ করে এবং কার্যক্রম সম্পর্কিত সুপারিশ যোগ করে।
3. **TravelManagerAgent** উভয় ইনপুটকে সংহত করে একটি চূড়ান্ত ভ্রমণ সংক্ষিপ্ত বিবরণ তৈরি করে।


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## প্রোডাকশন পরিবেশে A2A বোঝা

প্রোডাকশন পরিবেশে A2A প্রোটোকল শক্তিশালী ক্রস-সার্ভিস পরিস্থিতিগুলি উন্মোচন করে:

| ক্ষমতা | বর্ণনা |
|---|---|
| **ভিন্ন ফ্রেমওয়ার্কের মধ্যে আন্তঃঅপারেশন** | একটি ফ্রেমওয়ার্কে তৈরি এজেন্ট অন্য কোনো A2A-অনুগত ফ্রেমওয়ার্কে নির্মিত এজেন্টকে কাজ হস্তান্তর করতে পারে, যা প্রকৃতপক্ষে সংস্থা-অতিক্রম আন্তঃঅপারেবিলিটি সক্ষম করে। |
| **সেবা সীমানা** | এজেন্টগুলি আলাদা মাইক্রোসার্ভিস, ক্লাউড রিজিয়ন, বা এমনকি বিভিন্ন সংস্থায় অবস্থান করলেও সেগুলো নির্বিঘ্নে সহযোগিতা করতে পারে। |
| **গতি‍শীল আবিষ্কার** | একটি অর্কেস্ট্রেটর রানটাইমে একটি Agent Card রেজিস্ট্রিকে প্রশ্ন করে নির্দিষ্ট সাব-টাস্কের জন্য সবচেয়ে উপযুক্ত বিশেষজ্ঞ খুঁজে পেতে পারে। |
| **স্ট্রিমিং & পুশ নোটিফিকেশন** | A2A বাস্তব-সময়ের প্রগতি আপডেটের জন্য Server-Sent Events (SSE) এবং দীর্ঘসময় চলা কাজগুলির জন্য পুশ নোটিফিকেশন সমর্থন করে। |

উপর আমরা যে ওয়ার্কফ্লোটি তৈরি করেছি তা এই প্যাটার্নের একটি সরল, ইন-প্রসেস সংস্করণ। বাস্তব ডিপ্লয়মেন্টে প্রতিটি এজেন্ট একটি HTTP endpoint প্রকাশ করবে, একটি Agent Card প্রকাশ করবে, এবং A2A JSON-RPC প্রোটোকলের মাধ্যমে যোগাযোগ করবে।


## সারসংক্ষেপ

In this lesson you learned:

1. **A2A প্রোটোকল কী** — এজেন্ট-টু-এজেন্ট ডিসকভারি, মেসেজিং,
   এবং টাস্ক ম্যানেজমেন্টের জন্য একটি খোলা স্ট্যান্ডার্ড।
2. **কিভাবে বিশেষায়িত এজেন্ট তৈরি করবেন** — একটি কারেন্সি এক্সচেঞ্জ এজেন্ট, একটি অ্যাক্টিভিটি প্ল্যানার এজেন্ট,
   এবং একটি ট্রাভেল ম্যানেজার অর্কেস্ট্রেটর।
3. **কিভাবে এজেন্টগুলি একটি ওয়ার্কফ্লোতে সংযুক্ত করবেন** — `WorkflowBuilder` ব্যবহার করে ধারাবাহিক
   এজেন্টদের মধ্যে মেসেজ পাসিং মডেল করা।
4. **A2A প্রোডাকশনে কীভাবে কাজ করে** — ক্রস-ফ্রেমওয়ার্ক, ক্রস-সার্ভিস সহযোগিতা সক্ষম করা
   গতিশীল ডিসকভারি এবং স্ট্রিমিং আপডেটের মাধ্যমে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
অস্বীকৃতি:
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনুবাদ করা হয়েছে। আমরা যথাসাধ্য সঠিকতা বজায় রাখার চেষ্টা করি, তবু স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। সিদ্ধান্তমূলক বা নির্ভরযোগ্য উৎস হিসেবে মূল ভাষায় রচিত নথিকেই বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের ক্ষেত্রে পেশাদার মানব অনুবাদ গ্রহণ করার পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারে সৃষ্টি হওয়া কোনো ভুলবোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
